In [2]:
!pip install datasets pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:
from datasets import load_dataset

# Load LexGLUE datasets
ledgar = load_dataset("lex_glue", "ledgar")          # Clause classification
eurlex = load_dataset("lex_glue", "eurlex")          # Multi-label legal topics
unfair_tos = load_dataset("lex_glue", "unfair_tos")  # Fairness in Terms of Service


README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/3.31M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/3.44M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/147k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/218k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5532 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1607 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2275 [00:00<?, ? examples/s]

In [ ]:
ledgar_label_map = {
    "Confidentiality": "Confidentiality",
    "Non-Disparagement": "Confidentiality",
    "Termination": "Termination",
    "Severability": "Termination",
    "Death": "Termination",
    "Payments": "Payment",
    "Fees": "Payment",
    "Base Salary": "Payment",
    "Indemnifications": "Indemnification",
    "Indemnity": "Indemnification",
    "Jurisdictions": "Legal Governance",
    "Governing Laws": "Legal Governance",
    "Submission To Jurisdiction": "Legal Governance",
    "Consent To Jurisdiction": "Legal Governance",
    "Warranties": "Declarations",
    "Representations": "Declarations",
    "Intellectual Property": "IP & Rights",
    "Licenses": "IP & Rights",
    "Conflicts": "Miscellaneous",
    "Assignments": "Miscellaneous",
    "Modifications": "Miscellaneous",
    "Miscellaneous": "Miscellaneous",
    "Entire Agreements": "Miscellaneous",
}

eurlex_label_map = {
    1: "Legal Governance",  # Internal Market
    2: "Legal Governance",  # Agriculture
    3: "Business",          # Business and Competition
    4: "Consumers",         # Consumer Protection
    5: "Economy",           # Economic and Financial Affairs
    6: "Education",         # Education, Training, Youth
    7: "Employment",        # Employment and Labor
    8: "Environment",       # Environment and Climate
    9: "External Relations",# External and Foreign Policy
    10: "Social",           # Social Questions
    11: "Health",           # Public Health
}

unfair_label = "Fairness"


In [ ]:
from tqdm import tqdm

merged_data = []

# LEDGAR: already clause-based
for example in tqdm(ledgar["train"]):
    text = example["text"]
    label = ledgar["train"].features["label"].int2str(example["label"])
    mapped_label = ledgar_label_map.get(label, None)
    if mapped_label:
        merged_data.append({"text": text, "label": mapped_label})

# EUR-LEX: Break documents into sentences, assign major labels
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

label_names = eurlex["train"].features["labels"].feature.names

eurlex_subset = eurlex["train"].select(range(5000))

for example in tqdm(eurlex_subset):
    text = example["text"]
    sentences = sent_tokenize(text)
    labels = example["labels"]
    mapped_labels = [eurlex_label_map.get(l) for l in labels if l in eurlex_label_map]
    if mapped_labels:
        for sent in sentences:
            merged_data.append({"text": sent, "label": mapped_labels[0]})  # Use the first major label

100%|██████████| 60000/60000 [00:18<00:00, 3200.93it/s]
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
100%|██████████| 5000/5000 [00:05<00:00, 917.71it/s] 


In [ ]:
# UNFAIR-ToS: Use "text" field for the clause content
for example in tqdm(unfair_tos["train"]):
    text = example["text"]
    merged_data.append({"text": text, "label": unfair_label})


100%|██████████| 5532/5532 [00:00<00:00, 25793.43it/s]


In [ ]:
import random
random.shuffle(merged_data)

for i in range(5):
    print(merged_data[i])


{'text': '(64) Thirdly, if the Commission were to ban the scheme, the centres - or the groups to which they belong - would suffer significant losses if they were not able to take advantage of the remaining period of their current approval.', 'label': 'External Relations'}
{'text': 'The ‘Report of phytosanitary inspection’ shall be added as a supplement to the accompanying document as referred to in Article 2.', 'label': 'External Relations'}
{'text': 'your use of external links is therefore entirely at your own risk . \n', 'label': 'Fairness'}
{'text': 'The decision to refuse an application shall be set down in writing and sent to the applicant with the grounds for refusal.', 'label': 'Legal Governance'}
{'text': 'With a view to implementing measures contained in programmes as referred to in Article 6 of Regulation (EC) No 2826/2000, the Community trade federations or interbranch organisations that are representative of the sector(s) concerned shall submit programmes in response to cal

In [ ]:
from sklearn.utils import resample

df_balanced = pd.DataFrame()

for label in df['label'].unique():
    group = df[df['label'] == label]
    if len(group) > 2000:
        group = resample(group, n_samples=2000, random_state=42)
    df_balanced = pd.concat([df_balanced, group])

df_balanced = df_balanced.sample(frac=1).reset_index(drop=True)


In [ ]:
import pandas as pd
df = pd.DataFrame(merged_data)
print(df['label'].value_counts())


label
Environment           17574
Consumers             15714
Legal Governance       9985
Social                 8880
External Relations     7264
Fairness               5532
Health                 5459
Miscellaneous          4401
Employment             4143
Business               2815
Payment                2090
Termination            2086
Confidentiality        1375
Indemnification        1207
Declarations           1068
IP & Rights             473
Economy                 265
Education               122
Name: count, dtype: int64


In [ ]:
df.to_csv("merged_legal_dataset.csv", index=False)
df.to_json("merged_legal_dataset.json", orient="records", lines=True)


In [ ]:
from datasets import Dataset, DatasetDict
from huggingface_hub import notebook_login
from datasets import load_dataset

notebook_login()  # Login to Hugging Face account (first time only)

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Push to Hub
hf_dataset.push_to_hub("Kanishkagarwal6101/Legal_Document_Analyzer_mergedDataset")


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/91 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Document_Analyzer_mergedDataset/commit/9aab0b1be23d79d681dea9c1a12883125d0ba268', commit_message='Upload dataset', commit_description='', oid='9aab0b1be23d79d681dea9c1a12883125d0ba268', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Document_Analyzer_mergedDataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kanishkagarwal6101/Legal_Document_Analyzer_mergedDataset'), pr_revision=None, pr_num=None)

In [ ]:
from datasets import load_dataset

# Load from your Hugging Face hub
dataset = load_dataset("Kanishkagarwal6101/Legal_Document_Analyzer_mergedDataset")

# Inspect basic structure
print(dataset)
print("Sample:", dataset['train'][0])


README.md:   0%|          | 0.00/312 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 90453
    })
})
Sample: {'text': '(64) Thirdly, if the Commission were to ban the scheme, the centres - or the groups to which they belong - would suffer significant losses if they were not able to take advantage of the remaining period of their current approval.', 'label': 'External Relations'}


In [ ]:
from transformers import AutoTokenizer

# Use Legal-BERT
model_checkpoint = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )
    tokens["labels"] = example["label"]
    return tokens

# Tokenize and remove raw text
tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/90453 [00:00<?, ? examples/s]

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load HF dataset
dataset = load_dataset("Kanishkagarwal6101/Legal_Document_Analyzer_mergedDataset")
df = pd.DataFrame(dataset["train"])

# Define label remapping
label_remap = {
    "Payment": "Financial", "Fees": "Financial",
    "Termination": "Liability", "Indemnification": "Liability",
    "IP & Rights": "Legal Rights", "Declarations": "Legal Rights",
    "Confidentiality": "Protections", "Fairness": "Protections", "Consumers": "Protections",
    "Legal Governance": "Law", "Jurisdictions": "Law",
    "Employment": "Employment",
    "Social": "Miscellaneous", "Environment": "Miscellaneous",
    "Education": "Miscellaneous", "Economy": "Miscellaneous",
    "External Relations": "Miscellaneous", "Health": "Miscellaneous",
    "Business": "Miscellaneous", "Miscellaneous": "Miscellaneous"
}

label2id = {
    "Employment": 0,
    "Financial": 1,
    "Law": 2,
    "Legal Rights": 3,
    "Liability": 4,
    "Miscellaneous": 5,
    "Protections": 6
}

# Apply mapping
df["mapped_label"] = df["label"].map(label_remap)
df["label"] = df["mapped_label"].map(label2id)
df = df[["text", "label"]]


README.md:   0%|          | 0.00/312 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90453 [00:00<?, ? examples/s]

In [ ]:
# Downsample large classes to max 5000
MAX_SAMPLES = 5000
balanced_df = (
    df.groupby("label")
    .apply(lambda x: x.sample(min(len(x), MAX_SAMPLES), random_state=42))
    .reset_index(drop=True)
)


<ipython-input-4-58af80ecc31f>:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), MAX_SAMPLES), random_state=42))


In [ ]:
df_real = pd.read_csv("/content/predicted_real_world_clauses.csv")
df_real["label"] = df_real["Your Label"].map(label2id)
df_real = df_real[["Clause", "label"]]
df_real.columns = ["text", "label"]


In [ ]:
import pandas as pd
import random

# Re-load your manually labeled real contract clauses
df_original = pd.read_csv("/content/predicted_real_world_clauses.csv")

# Synonym substitution rules
synonyms = {
    "Agreement": "Contract",
    "Company": "Corporation",
    "Consultant": "Service Provider",
    "shall": "will",
    "may": "might",
    "submit": "send",
    "payment": "compensation",
    "terminate": "end",
    "confidential": "private",
    "liable": "responsible",
    "fees": "charges"
}

# Clause variation functions
def add_synonyms(text):
    for word, replacement in synonyms.items():
        text = text.replace(word, replacement)
    return text

def add_ending_noise(text):
    endings = [
        " This clause is binding and enforceable.",
        " This provision applies globally.",
        " Subject to negotiation in good faith.",
        " As required by applicable law.",
        " With mutual written consent."
    ]
    return text + random.choice(endings)

def compress_sentence(text):
    words = text.split()
    if len(words) > 20:
        return " ".join(words[:10]) + " ... " + " ".join(words[-10:])
    return text

# Generate 9 variations (1 original + 9 augmentations = 10)
synthetic_data = []

for _ in range(9):
    for _, row in df_original.iterrows():
        text = row["Clause"]
        label = row["Your Label"]
        transform = random.choice([add_synonyms, add_ending_noise, compress_sentence])
        new_text = transform(text)
        synthetic_data.append({"text": new_text.strip(), "label": label})

# Create dataframe
df_augmented = pd.DataFrame(synthetic_data)
df_augmented.head()

,text,label
0,CONSULTING SERVICES AGREEMENT\nThis Consulting...,Miscellaneous
1,"Tech Solutions Inc., a corporation registered ...",Miscellaneous
2,1.1 Scope of Work: Consultant shall provide st...,Employment
3,2.1 Fees: In consideration of the services pro...,Financial
4,3.1 Definition of Confidential Information: Bo...,Protections


In [ ]:
# Use df_augmented if already in memory — otherwise rerun generation block
df_augmented["label"] = df_augmented["label"].map(label2id)
df_augmented = df_augmented[["text", "label"]]


In [ ]:
df_merged = pd.concat([balanced_df, df_real, df_augmented], ignore_index=True)
df_merged = df_merged.sample(frac=1, random_state=42).reset_index(drop=True)

print("✅ Final training samples:", len(df_merged))
df_merged.head()


✅ Final training samples: 26187


,text,label
0,(6)\nThe EMN should avoid duplicating the work...,0
1,(8)\nThe Alpine Convention together with its i...,5
2,Either (a) on the date that is the Business Da...,3
3,(13)\nThe production of specific Community sta...,2
4,(e) Appropriate remuneration for the capital\n...,5


In [ ]:
from datasets import Dataset

# Convert merged dataframe to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df_merged, preserve_index=False)
!huggingface-cli login
hf_dataset.push_to_hub("Kanishkagarwal6101/Legal_Document_Analyzer_augmented", private=False)




    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: write

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/27 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/310 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Document_Analyzer_augmented/commit/b4f2902caf40f3602e1cf193be0dc6c1c96a23b2', commit_message='Upload dataset', commit_description='', oid='b4f2902caf40f3602e1cf193be0dc6c1c96a23b2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Document_Analyzer_augmented', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kanishkagarwal6101/Legal_Document_Analyzer_augmented'), pr_revision=None, pr_num=None)

In [4]:
from datasets import load_dataset, Dataset
import pandas as pd
import random

# ✅ Load Hugging Face dataset
hf_dataset = load_dataset("Kanishkagarwal6101/Legal_Document_Analyzer_augmented")
df_hf = pd.DataFrame(hf_dataset["train"])[["text", "label"]]

# ✅ Load real contract CSV
df_real = pd.read_csv("/content/smart_ensemble_employment_eval.csv")[["Clause", "Your Label"]]
df_real.columns = ["text", "label"]

# ✅ Augmentation rules
synonyms = {
    "Employee": "Staff Member", "Employer": "Company", "Contract": "Agreement",
    "shall": "will", "may": "might", "position": "role", "duties": "responsibilities",
    "terminate": "end", "confidential": "private", "liable": "responsible"
}

def apply_synonyms(text):
    for word, sub in synonyms.items():
        text = text.replace(word, sub)
    return text

def compress_text(text):
    words = text.split()
    if len(words) > 20:
        return " ".join(words[:10]) + " ... " + " ".join(words[-10:])
    return text

def add_suffix(text):
    suffixes = [
        " This clause is governed by HR policy.",
        " Refer to company manual for exceptions.",
        " Subject to change based on business needs.",
        " Applicable within employment jurisdiction.",
        " As outlined in the role handbook."
    ]
    return text + random.choice(suffixes)

# ✅ Generate 9 augmentations
augmented_rows = []
for _ in range(9):
    for _, row in df_real.iterrows():
        method = random.choice([apply_synonyms, compress_text, add_suffix])
        augmented_rows.append({
            "text": method(row["text"]).strip(),
            "label": row["label"]
        })
df_augmented = pd.DataFrame(augmented_rows)

# ✅ Merge everything
df_final = pd.concat([df_hf, df_real, df_augmented], ignore_index=True)

# ✅ Convert labels to integers
label2id = {
    "Employment": 0,
    "Financial": 1,
    "Law": 2,
    "Legal Rights": 3,
    "Liability": 4,
    "Miscellaneous": 5,
    "Protections": 6
}
df_final["label"] = df_final["label"].map(label2id)

# ✅ Upload to Hugging Face (overwrite existing)
hf_dataset_final = Dataset.from_pandas(df_final, preserve_index=False)

!huggingface-cli login  # Run this once if not logged in
hf_dataset_final.push_to_hub("Kanishkagarwal6101/Legal_Document_Analyzer_augmented", private=False)



    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: write).
The token `nlpprojectdataset` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `nlppro

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/27 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Document_Analyzer_augmented/commit/249d65c9813a56ea7c7ed42d83c254c93525722d', commit_message='Upload dataset', commit_description='', oid='249d65c9813a56ea7c7ed42d83c254c93525722d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Document_Analyzer_augmented', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kanishkagarwal6101/Legal_Document_Analyzer_augmented'), pr_revision=None, pr_num=None)

In [6]:
from datasets import load_dataset, Dataset
import pandas as pd

# ✅ Load dataset
ds = load_dataset("Kanishkagarwal6101/Legal_Document_Analyzer_augmented")
df = pd.DataFrame(ds["train"])

# ✅ Mapping
label2id = {
    "Employment": 0,
    "Financial": 1,
    "Law": 2,
    "Legal Rights": 3,
    "Liability": 4,
    "Miscellaneous": 5,
    "Protections": 6
}

# ✅ Fix labels
def fix_label(val):
    if isinstance(val, str):
        return label2id.get(val.strip(), None)
    elif isinstance(val, (int, float)) and not pd.isna(val):
        return int(val)
    return None

df["label"] = df["label"].apply(fix_label)

# ✅ Drop only rows where label is still invalid
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

print("✅ Final size after full label correction:", df.shape)

# ✅ Upload clean dataset
hf_cleaned = Dataset.from_pandas(df, preserve_index=False)
hf_cleaned.push_to_hub("Kanishkagarwal6101/Legal_Analyzer_Final", private=False)


✅ Final size after full label correction: (330, 2)


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/302 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Final/commit/dfab022d8a27cc53fd2ba913355ce338d90c3dac', commit_message='Upload dataset', commit_description='', oid='dfab022d8a27cc53fd2ba913355ce338d90c3dac', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Final', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kanishkagarwal6101/Legal_Analyzer_Final'), pr_revision=None, pr_num=None)

In [7]:
from datasets import load_dataset
import pandas as pd

# Load current dataset
ds = load_dataset("Kanishkagarwal6101/Legal_Document_Analyzer_augmented")
df = pd.DataFrame(ds["train"])

# Check all unique label types
print("🔍 Sample label values:", df["label"].unique()[:20])
print("\n🧮 Label types count:\n", df["label"].apply(lambda x: str(type(x))).value_counts())


🔍 Sample label values: [nan  0.  1.  5.  6.  3.  4.]

🧮 Label types count:
 label
<class 'float'>    26517
Name: count, dtype: int64


In [8]:
from datasets import load_dataset, Dataset
import pandas as pd

# ✅ Load full dataset
ds = load_dataset("Kanishkagarwal6101/Legal_Document_Analyzer_augmented")
df = pd.DataFrame(ds["train"])

# ✅ Drop rows with null labels
df = df.dropna(subset=["label"])

# ✅ Convert float to int labels
df["label"] = df["label"].astype(int)

print("✅ Cleaned dataset shape:", df.shape)

# ✅ Convert and upload to final dataset
hf_clean = Dataset.from_pandas(df, preserve_index=False)
hf_clean.push_to_hub("Kanishkagarwal6101/Legal_Analyzer_Final", private=False)


✅ Cleaned dataset shape: (330, 2)


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Final/commit/dfab022d8a27cc53fd2ba913355ce338d90c3dac', commit_message='Upload dataset', commit_description='', oid='dfab022d8a27cc53fd2ba913355ce338d90c3dac', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Final', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kanishkagarwal6101/Legal_Analyzer_Final'), pr_revision=None, pr_num=None)

In [9]:
from datasets import load_dataset
import pandas as pd

# Load the dataset
ds = load_dataset("Kanishkagarwal6101/Legal_Document_Analyzer_mergedDataset")
df = pd.DataFrame(ds["train"])

# Show unique label values
print("🧾 Unique label values in the dataset:")
print(sorted(df["label"].dropna().unique()))

README.md:   0%|          | 0.00/312 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90453 [00:00<?, ? examples/s]

🧾 Unique label values in the dataset:
['Business', 'Confidentiality', 'Consumers', 'Declarations', 'Economy', 'Education', 'Employment', 'Environment', 'External Relations', 'Fairness', 'Health', 'IP & Rights', 'Indemnification', 'Legal Governance', 'Miscellaneous', 'Payment', 'Social', 'Termination']


In [10]:
from datasets import load_dataset, Dataset
import pandas as pd

# ✅ Step 1: Load the 330-row dataset from Hugging Face
ds = load_dataset("Kanishkagarwal6101/Legal_Analyzer_Final")
df = pd.DataFrame(ds["train"])

# ✅ Step 2: Reverse map int → string using old 7-class logic
old_id2label = {
    0: "Employment",
    1: "Financial",
    2: "Law",
    3: "Legal Rights",
    4: "Liability",
    5: "Miscellaneous",
    6: "Protections"
}
df["label"] = df["label"].map(old_id2label)

# ✅ Step 3: Remap to new 18-class IDs
new_label2id = {
    "Employment": 6,
    "Financial": 15,
    "Law": 13,
    "Legal Rights": 11,
    "Liability": 17,
    "Miscellaneous": 14,
    "Protections": 1
}
df["label"] = df["label"].map(new_label2id)
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

# ✅ Step 4: Upload back to same dataset
hf_cleaned = Dataset.from_pandas(df, preserve_index=False)
hf_cleaned.push_to_hub("Kanishkagarwal6101/Legal_Analyzer_Final", private=False)


train-00000-of-00001.parquet:   0%|          | 0.00/30.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/330 [00:00<?, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Final/commit/36c4aff9e11a649a7fc5a906270339d4df9db35f', commit_message='Upload dataset', commit_description='', oid='36c4aff9e11a649a7fc5a906270339d4df9db35f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Final', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kanishkagarwal6101/Legal_Analyzer_Final'), pr_revision=None, pr_num=None)

In [11]:
from datasets import load_dataset, Dataset
import pandas as pd

# ✅ Step 1: Load the merged dataset
ds_merged = load_dataset("Kanishkagarwal6101/Legal_Document_Analyzer_mergedDataset")
df = pd.DataFrame(ds_merged["train"])

# ✅ Step 2: Apply 18-class label mapping
label2id = {
    "Business": 0,
    "Confidentiality": 1,
    "Consumers": 2,
    "Declarations": 3,
    "Economy": 4,
    "Education": 5,
    "Employment": 6,
    "Environment": 7,
    "External Relations": 8,
    "Fairness": 9,
    "Health": 10,
    "IP & Rights": 11,
    "Indemnification": 12,
    "Legal Governance": 13,
    "Miscellaneous": 14,
    "Payment": 15,
    "Social": 16,
    "Termination": 17
}
df["label"] = df["label"].map(label2id)
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

# ✅ Step 3: Downsample (max 3000 per class)
df_balanced = (
    df.groupby("label")
    .apply(lambda x: x.sample(n=min(len(x), 3000), random_state=42))
    .reset_index(drop=True)
)

print("✅ Final dataset shape after downsampling:", df_balanced.shape)

# ✅ Step 4: Save locally (optional)
df_balanced.to_csv("legal_downsampled.csv", index=False)

# ✅ Step 5: Convert to Hugging Face Dataset and upload
ds_balanced = Dataset.from_pandas(df_balanced, preserve_index=False)
ds_balanced.push_to_hub("Kanishkagarwal6101/Legal_Analyzer_Downsampled", private=False)


<ipython-input-11-db34f0c81da4>:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 3000), random_state=42))


✅ Final dataset shape after downsampling: (38501, 2)


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/39 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Downsampled/commit/6de536d8a6a17c0e94cc80d09c4e83c1ce73332b', commit_message='Upload dataset', commit_description='', oid='6de536d8a6a17c0e94cc80d09c4e83c1ce73332b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Downsampled', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kanishkagarwal6101/Legal_Analyzer_Downsampled'), pr_revision=None, pr_num=None)

In [12]:
from datasets import load_dataset, Dataset
import pandas as pd

# ✅ Step 1: Load downsampled dataset (main body)
ds_main = load_dataset("Kanishkagarwal6101/Legal_Analyzer_Downsampled")
df_main = pd.DataFrame(ds_main["train"])

# ✅ Step 2: Load fixed 330-row dataset
ds_aug = load_dataset("Kanishkagarwal6101/Legal_Analyzer_Final")
df_aug = pd.DataFrame(ds_aug["train"])

# ✅ Step 3: Merge and shuffle
df_merged = pd.concat([df_main, df_aug], ignore_index=True).sample(frac=1, random_state=42)

print("✅ Final merged shape:", df_merged.shape)

# ✅ Step 4: Convert and upload to same dataset name
final_ds = Dataset.from_pandas(df_merged, preserve_index=False)
final_ds.push_to_hub("Kanishkagarwal6101/Legal_Analyzer_Final", private=False)


README.md:   0%|          | 0.00/310 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/6.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38501 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/302 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/30.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/330 [00:00<?, ? examples/s]

✅ Final merged shape: (38831, 2)


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/39 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Final/commit/4bb1e92a9ef042efec67ce990467db5e9f0566f2', commit_message='Upload dataset', commit_description='', oid='4bb1e92a9ef042efec67ce990467db5e9f0566f2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Kanishkagarwal6101/Legal_Analyzer_Final', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Kanishkagarwal6101/Legal_Analyzer_Final'), pr_revision=None, pr_num=None)